<a href="https://colab.research.google.com/github/ednnaldodev/Modulo4_PRF.ipynb-/blob/aula_2_17%2F08/Modulo4_PRF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
import unicodedata
# Vamos conectar ao Google Drive pela praticidade!
from google.colab import drive
drive.mount ('/content/drive')

Mounted at /content/drive


In [6]:
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

In [7]:
# Estrutura padrão do projeto
RAIZ = Path ("/content/drive/MyDrive/Colab Notebooks/prf_2025")
PASTAS = [
"dados_brutos", "dados_tratados",
"notebooks", "sql", "dashboards",
"relatorios", "apresentacao", "logs"
]
for pasta in PASTAS:
  (RAIZ / pasta).mkdir(
parents=True, exist_ok=True)
print("Pastas verificadas/criadas:")
for pasta in PASTAS:
  print("-", RAIZ / pasta)

Pastas verificadas/criadas:
- /content/drive/MyDrive/Colab Notebooks/prf_2025/dados_brutos
- /content/drive/MyDrive/Colab Notebooks/prf_2025/dados_tratados
- /content/drive/MyDrive/Colab Notebooks/prf_2025/notebooks
- /content/drive/MyDrive/Colab Notebooks/prf_2025/sql
- /content/drive/MyDrive/Colab Notebooks/prf_2025/dashboards
- /content/drive/MyDrive/Colab Notebooks/prf_2025/relatorios
- /content/drive/MyDrive/Colab Notebooks/prf_2025/apresentacao
- /content/drive/MyDrive/Colab Notebooks/prf_2025/logs


In [8]:
# Parâmetros do projeto
ARQUIVO_BRUTO = Path("dados_brutos/acidentes2025.csv")
ARQUIVO_BASE_ANALITICA = Path ("dados_tratados/base_analitica_completa.csv")
ARQUIVO_BASE_MODELAVEL = Path ("dados_tratados/base_modelavel_preliminar.csv")
ARQUIVO_DICIONARIO = Path ("dados_tratados/dicionario_variaveis_modulo4.csv")
ARQUIVO_DECISOES = Path ("logs/decisoes_tratamento_modulo4.md")
ARQUIVO_README = Path ("README.md")
SEPARADOR = ";"
ENCODING_ENTRADA = "latin1"
ENCODING_SAIDA = "utf-8-sig"

In [9]:
# Leitura CSV da PRF (com Fallback)
def ler_csv_prf(caminho, sep=";", encodings=("utf-8-sig", "utf-8", "latin1")):
  ultimo_erro = None
  for enc in encodings:
    try:
      print(f"Tentando leitura com encodin={enc}...")
      return pd.read_csv(caminho, sep=sep, encoding=enc, low_memory=False)
    except Exception as erro:
      ultimo_erro = erro
      print(f"Falhou com {enc}: {erro}")
    raise ultimo_erro

In [10]:
df = ler_csv_prf(RAIZ / ARQUIVO_BRUTO, sep=SEPARADOR)
df.head(5)

Tentando leitura com encodin=utf-8-sig...


,id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,pessoas,mortos,feridos_leves,feridos_graves,ilesos,ignorados,feridos,veiculos,latitude,longitude,regional,delegacia,uop
0,652493,2025-01-01,quarta-feira,06:20:00,SP,116,225,GUARULHOS,Reação tardia ou ineficiente do condutor,Tombamento,Com Vítimas Feridas,Pleno dia,Decrescente,Céu Claro,Múltipla,Reta;Declive,Sim,2,0,1,0,0,1,1,2,"-23,48586772","-46,54075317",SPRF-SP,DEL01-SP,UOP01-DEL01-SP
1,652519,2025-01-01,quarta-feira,07:50:00,CE,116,"546,2",PENAFORTE,Pista esburacada,Colisão frontal,NaN,Pleno dia,Crescente,Céu Claro,Simples,Reta,Não,6,1,1,0,1,4,1,6,"-7,812288","-39,08333306",SPRF-CE,DEL05-CE,UOP03-DEL05-CE
2,652522,2025-01-01,quarta-feira,08:45:00,PR,369,"88,2",CORNELIO PROCOPIO,Reação tardia ou ineficiente do condutor,Colisão traseira,Com Vítimas Feridas,Pleno dia,Crescente,Sol,Dupla,Reta;Aclive,Sim,5,0,3,0,2,0,3,2,"-23,182565","-50,637228",SPRF-PR,DEL07-PR,UOP05-DEL07-PR
3,652544,2025-01-01,quarta-feira,11:00:00,PR,116,74,CAMPINA GRANDE DO SUL,Reação tardia ou ineficiente do condutor,Saída de leito carroçável,Com Vítimas Feridas,Pleno dia,Crescente,Céu Claro,Dupla,Reta,Não,5,0,1,0,4,0,1,2,"-25,36517687","-49,04223028",SPRF-PR,DEL01-PR,UOP02-DEL01-PR
4,652549,2025-01-01,quarta-feira,09:30:00,MG,251,471,FRANCISCO SA,Velocidade Incompatível,Colisão frontal,Com Vítimas Feridas,Pleno dia,Decrescente,Chuva,Simples,Curva;Declive,Não,5,0,1,1,1,2,2,4,"-16,46801304","-43,43121303",SPRF-MG,DEL12-MG,UOP01-DEL12-MG


In [11]:
# Padronizaçao dos nome das colunas
def normalizar_nome_coluna(nome):
  nome = str(nome).strip().lower()
  nome = unicodedata.normalize("NFKD", nome).encode("ascii", "ignore").decode("utf-8")
  nome = nome.replace(" ", "_").replace("-", "_").replace("/", "_")
  while "__" in nome:
    nome = nome.replace("__", "_")
  return nome.strip("_")
df.columns = [normalizar_nome_coluna(c) for c in df.columns]

# Compatibilização de grafias possíveis

df = df.rename(columns={"condicao_metereologica": "condicao_meteorologica"})
print(df.columns.tolist())

['id', 'data_inversa', 'dia_semana', 'horario', 'uf', 'br', 'km', 'municipio', 'causa_acidente', 'tipo_acidente', 'classificacao_acidente', 'fase_dia', 'sentido_via', 'condicao_meteorologica', 'tipo_pista', 'tracado_via', 'uso_solo', 'pessoas', 'mortos', 'feridos_leves', 'feridos_graves', 'ilesos', 'ignorados', 'feridos', 'veiculos', 'latitude', 'longitude', 'regional', 'delegacia', 'uop']


In [12]:
# Compatibilização de grafias possíveis
df = df.rename(columns={"condicao_metereologica": "condicao_meteorologica"})
print(df.columns.tolist())

['id', 'data_inversa', 'dia_semana', 'horario', 'uf', 'br', 'km', 'municipio', 'causa_acidente', 'tipo_acidente', 'classificacao_acidente', 'fase_dia', 'sentido_via', 'condicao_meteorologica', 'tipo_pista', 'tracado_via', 'uso_solo', 'pessoas', 'mortos', 'feridos_leves', 'feridos_graves', 'ilesos', 'ignorados', 'feridos', 'veiculos', 'latitude', 'longitude', 'regional', 'delegacia', 'uop']


In [14]:
#Conferir Colunas Esperadas
colunas_esperadas = [
"data_inversa", "dia_semana", "horario", "uf", "br", "municipio",
"causa_acidente", "tipo_acidente", "classificacao_acidente",
"fase_dia", "condicao_meteorologica", "tipo_pista", "tracado_via",
"uso_solo", "pessoas", "mortos", "feridos_leves",
"feridos_graves", "feridos", "veiculos"
]

faltantes = [c for c in colunas_esperadas if c not in df.columns]
print("Colunas faltantes:", faltantes)

if faltantes:
  print("Atenção: ajuste nomes ou confirme o dicionário oficial da PRF usado no arquivo. ")

Colunas faltantes: []


In [49]:
# Tipos de dados e memória utilizada
#O que observar
#Tipos inferidos automaticamente pelo pandas
#Uso de memória da base
#Colunas numéricas lidas como texto
#Datas ainda no tipo object

df.info(memory_usage="deep")
resumo_tipos = (
df.dtypes.astype(str)
.value_counts()
.rename_axis("tipo")
.reset_index(name="qtd_colunas")
)
display(resumo_tipos)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72529 entries, 0 to 72528
Data columns (total 33 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      72529 non-null  int64  
 1   data_inversa            72529 non-null  object 
 2   dia_semana              72529 non-null  object 
 3   horario                 72529 non-null  object 
 4   uf                      72529 non-null  object 
 5   br                      72529 non-null  int64  
 6   km                      45854 non-null  float64
 7   municipio               72529 non-null  object 
 8   causa_acidente          72529 non-null  object 
 9   tipo_acidente           72529 non-null  object 
 10  classificacao_acidente  72529 non-null  object 
 11  fase_dia                72529 non-null  object 
 12  sentido_via             72529 non-null  object 
 13  condicao_meteorologica  72529 non-null  object 
 14  tipo_pista              72529 non-null

,tipo,qtd_colunas
0,object,21
1,int64,10
2,float64,1
3,int32,1


In [48]:
# Diagnóstico de valores ausentes
#Conceitos Importantes
#Quantidade e percentual de nulos por coluna
#Priorizar campos analiticamente relevantes
#Não confundir nulo com zero
#Valor ausente = falta de registro. Zero é uma informaçãoválida em campos como mortos.

nulos = pd.DataFrame({
"qtd_nulos": df.isna().sum(),
"perc_nulos": df.isna().mean() * 100
}).sort_values(
"perc_nulos", ascending=False)
display(nulos[nulos["qtd_nulos"] > 0])

,qtd_nulos,perc_nulos
km,26675,36.778392
uop,38,0.052393
delegacia,22,0.030333
regional,2,0.002758


In [17]:
# Diagnóstico e remoção de duplicidades
qtd_duplicadas = df.duplicated().sum()

print("Duplicidades exatas:", qtd_duplicadas)
if qtd_duplicadas > 0:
  df = df.drop_duplicates().copy()

#Regras para duplicidades
#Identificar duplicidade exata (todas as colunas iguais)
#Remoção controlada e rastreável
#Diferenciar de eventos parecidos mas distintos
#Registrar a decisão no notebook

print("Duplicidades removidas.")
print("Nova dimensão:", df.shape)

Duplicidades exatas: 0
Duplicidades removidas.
Nova dimensão: (72529, 30)


In [18]:
# Cardinalidade das variáveis categóricas
categoricas = df.select_dtypes(
include="object").columns
cardinalidade = (
df[categoricas]
.nunique(dropna=True)
.sort_values(ascending=False)

#O que é cardinalidade? Número de categorias únicas por variável.
#Use para antecipardesafios de análise e modelagem.
#Município tende a ter alta cardinalidade
#Causa e tipo exigem cuidado especial
#Categorias raras podem distorcer análises

.reset_index()
)
cardinalidade.columns = [
  "variavel","qtd_categorias"]
display(cardinalidade.head(30))

,variavel,qtd_categorias
0,latitude,69294
1,longitude,69237
2,km,7655
3,municipio,1844
4,horario,1412
5,tracado_via,605
6,uop,395
7,data_inversa,365
8,delegacia,153
9,causa_acidente,69


In [19]:
#Converter Colunas Numéricas
colunas_numericas = [
"br","km","pessoas","mortos","feridos",
"feridos_leves","feridos_graves",
"ilesos","ignorados","veiculos"
]

#Colunas a converter
#Contagens de vítimas: mortos, feridos, feridos_leves,
#feridos_graves
#Localização: br, km
#Totais: pessoas, veiculos


for coluna in colunas_numericas:
 if coluna in df.columns:
  df[coluna] = pd.to_numeric(df[coluna], errors="coerce")

print(df[[c for c in colunas_numericas if c in df.columns]].dtypes)

br                  int64
km                float64
pessoas             int64
mortos              int64
feridos             int64
feridos_leves       int64
feridos_graves      int64
ilesos              int64
ignorados           int64
veiculos            int64
dtype: object


In [27]:
#Tratar horario e criar flags de turno
horario_limpo = df["horario"].astype(str).str.strip()
df["hora"] = pd.to_datetime(horario_limpo, format="%H:%M:%S", errors="coerce").dt.hour

def classificar_turno(hora):
  if pd.isna(hora): return "IGNORADO"
  if 0 <= hora < 6: return "NOITE"
  if 6 <= hora < 12: return "MANHÃ"
  if 12 <= hora < 18: return "TARDE"

  return "NOIDE"

df["turno"] = df["hora"].apply(classificar_turno)
df.head(df)


,id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_meteorologica,tipo_pista,tracado_via,uso_solo,pessoas,mortos,feridos_leves,feridos_graves,ilesos,ignorados,feridos,veiculos,latitude,longitude,regional,delegacia,uop,hora,turno
0,652493,2025-01-01,quarta-feira,06:20:00,SP,116,225.0,GUARULHOS,Reação tardia ou ineficiente do condutor,Tombamento,Com Vítimas Feridas,Pleno dia,Decrescente,Céu Claro,Múltipla,Reta;Declive,Sim,2,0,1,0,0,1,1,2,"-23,48586772","-46,54075317",SPRF-SP,DEL01-SP,UOP01-DEL01-SP,6,MANHÃ
1,652519,2025-01-01,quarta-feira,07:50:00,CE,116,NaN,PENAFORTE,Pista esburacada,Colisão frontal,NaN,Pleno dia,Crescente,Céu Claro,Simples,Reta,Não,6,1,1,0,1,4,1,6,"-7,812288","-39,08333306",SPRF-CE,DEL05-CE,UOP03-DEL05-CE,7,MANHÃ
2,652522,2025-01-01,quarta-feira,08:45:00,PR,369,NaN,CORNELIO PROCOPIO,Reação tardia ou ineficiente do condutor,Colisão traseira,Com Vítimas Feridas,Pleno dia,Crescente,Sol,Dupla,Reta;Aclive,Sim,5,0,3,0,2,0,3,2,"-23,182565","-50,637228",SPRF-PR,DEL07-PR,UOP05-DEL07-PR,8,MANHÃ
3,652544,2025-01-01,quarta-feira,11:00:00,PR,116,74.0,CAMPINA GRANDE DO SUL,Reação tardia ou ineficiente do condutor,Saída de leito carroçável,Com Vítimas Feridas,Pleno dia,Crescente,Céu Claro,Dupla,Reta,Não,5,0,1,0,4,0,1,2,"-25,36517687","-49,04223028",SPRF-PR,DEL01-PR,UOP02-DEL01-PR,11,MANHÃ
4,652549,2025-01-01,quarta-feira,09:30:00,MG,251,471.0,FRANCISCO SA,Velocidade Incompatível,Colisão frontal,Com Vítimas Feridas,Pleno dia,Decrescente,Chuva,Simples,Curva;Declive,Não,5,0,1,1,1,2,2,4,"-16,46801304","-43,43121303",SPRF-MG,DEL12-MG,UOP01-DEL12-MG,9,MANHÃ


In [32]:
#Criar Faixa Horária
def criar_faixa_horaria(hora):
  if pd.isna(hora):
    return "IGNORADO" #<- redundante trata NaN

#Blocos de 3 horas
#Reduz granularidade excessiva do horário
#Ajuda a identificar padrões temporais
#Útil para modelo explicável
#Exemplo: 00h-02h, 03h-05h...

  inicio = int(hora // 3) * 3
  fim = inicio + 2
  return f"{inicio:02d}h-{fim:02d}h"

df["faixa_horaria"] = df["hora"].apply(criar_faixa_horaria)  # <-não descarte valores NaN da contagem, mostre eles também no resultado

display(df["faixa_horaria"].value_counts(dropna=False).sort_index())


,count
faixa_horaria,
00h-02h,3959
03h-05h,4948
06h-08h,11517
09h-11h,9342
12h-14h,9678
15h-17h,12624
18h-20h,13473
21h-23h,6988


In [47]:
#Tratar Nulos Categóricos
categoricas_importantes = [
"uf","municipio","causa_acidente","tipo_acidente","fase_dia","tipo_pista",
"tracado_via","uso_solo","classificacao_acidente","dia_semana","condicao_meteorologica"
]

#Usar "IGNORADO" em campos explicativos
#Evitar perda de linhas por nulos
#Preservar a informação de ausência
#Documentar a decisão no notebook

for coluna in categoricas_importantes:
  if coluna in df.columns:
    df[coluna] = df[coluna].fillna("IGNORADO")

print(df[categoricas_importantes].isna().sum().sort_values(ascending=False))

uf                        0
municipio                 0
causa_acidente            0
tipo_acidente             0
fase_dia                  0
tipo_pista                0
tracado_via               0
uso_solo                  0
classificacao_acidente    0
dia_semana                0
condicao_meteorologica    0
dtype: int64


In [53]:
#Tratar Nulos Numéricos de Contagem
#Decisão metodológica
#Contagens de vítimas: mortos, feridos, pessoas, veiculos
#Zero como hipótese operacional conservadora
#Atenção à semântica de cada coluna
#Registrar a regra adotada no notebook

contagens_vitimas = [
"mortos","feridos","feridos_leves",
"feridos_graves","pessoas","veiculos"
]

for coluna in contagens_vitimas:
  if coluna in df.columns:
   df[coluna] = df[coluna].fillna(0)

print(df[[c for c in contagens_vitimas if c in df.columns]].isna().sum())

mortos            0
feridos           0
feridos_leves     0
feridos_graves    0
pessoas           0
veiculos          0
dtype: int64


In [52]:
#Criar Variável-Alvo: acidente_fatal
#Regra
#1 quando mortos >= 1
#0 quando mortos = 0


df["acidente_fatal"] = np.where(
  df["mortos"] >= 1, 1, 0)


validacao_alvo = (
  df["acidente_fatal"]
  .value_counts(dropna=False)
  .rename_axis("acidente_fatal")
  .reset_index(name="qtd"))
validacao_alvo["perc"] = (
  validacao_alvo["qtd"] /
  validacao_alvo["qtd"].sum() * 100)
display(validacao_alvo)

,acidente_fatal,qtd,perc
0,0,67319,92.816666
1,1,5210,7.183334


In [54]:
# Teste lógico da variável-alvo
#Validar Logicamente o Alvo O que testar
#Detectar inversões na regra (0 onde deveria ser 1)
#Detectar nulos indevidos no alvo
#Usar assert para garantir zero violações

violacoes = df.loc[
((df["mortos"] >= 1) &
(df["acidente_fatal"] != 1)) |
((df["mortos"] == 0) &
(df["acidente_fatal"] != 0))
]

print("Violações da regra do alvo:",
len(violacoes))

assert len(violacoes) == 0, \
"Há erro na criação de acidente_fatal."

Violações da regra do alvo: 0


In [55]:
#Criar BR Formatada e Chave de Localidade
#Variáveis de identificação analítica
#br_formatada: formato BR-000 ou BR-IGNORADA
#chave_localidade: UF + município + BR formatada
#Facilita filtros e agrupamentos no Power BI

def formatar_br(valor):
  if pd.isna(valor) or valor == 0:
    return "BR-IGNORADA"
    return f"BR-{int(valor):03d}"

df["br_formatada"] = df["br"].apply(formatar_br)

df["chave_localidade"] = (
  df["uf"].astype(str) + "_" +
  df["municipio"].astype(str) + "_" +
  df["br_formatada"].astype(str)
)

display(df[["uf","municipio","br","br_formatada","chave_localidade"]].head())


,uf,municipio,br,br_formatada,chave_localidade
0,SP,GUARULHOS,116,None,SP_GUARULHOS_None
1,CE,PENAFORTE,116,None,CE_PENAFORTE_None
2,PR,CORNELIO PROCOPIO,369,None,PR_CORNELIO PROCOPIO_None
3,PR,CAMPINA GRANDE DO SUL,116,None,PR_CAMPINA GRANDE DO SUL_None
4,MG,FRANCISCO SA,251,None,MG_FRANCISCO SA_None


In [60]:
#Checagens Rápidas Após Transformação

#SHAPE > Linhas e Colunas Verificar dimensões após todas as transformações.
#SUM > Acidentes Fatais Total de registros com acidente_fatal = 1.
#MEAN > Taxa de Fatalidade Proporção global de acidentes fatais na base.

checagens = {
  "linhas": len(df),
  "colunas": df.shape[1],
  "acidentes_fatais": int(df["acidente_fatal"].sum()),
  "taxa_fatalidade": float(df["acidente_fatal"].mean()),
  "total_mortos": int(df["mortos"].sum()),
}

checagens


{'linhas': 72529,
 'colunas': 36,
 'acidentes_fatais': 5210,
 'taxa_fatalidade': 0.07183333563126473,
 'total_mortos': 6043}

Ranking Rápido de Categorias


In [61]:
#Ranking Rápido de Categorias Objetivo
#Conferir se as categorias estão coerentes
#Identificar causas e tipos mais frequentes
#Preparação para a EDA formal no Módulo 5

def ranking_categoria(base, coluna, n=10):
  return (
  base[coluna]
  .value_counts(dropna=False)
  .head(n)
  .rename_axis(coluna)
  .reset_index(name="qtd")
)


display(ranking_categoria(df, "causa_acidente", 10))

display(ranking_categoria(df, "tipo_acidente", 10))

,causa_acidente,qtd
0,Ausência de reação do condutor,11469
1,Reação tardia ou ineficiente do condutor,10799
2,Acessar a via sem observar a presença dos outr...,7097
3,Condutor deixou de manter distância do veículo...,4413
4,Velocidade Incompatível,4088
5,Manobra de mudança de faixa,4016
6,Ingestão de álcool pelo condutor,3685
7,Demais falhas mecânicas ou elétricas,3385
8,Transitar na contramão,2475
9,Condutor Dormindo,2116


,tipo_acidente,qtd
0,Colisão traseira,14360
1,Saída de leito carroçável,10209
2,Colisão transversal,9306
3,Colisão lateral mesmo sentido,7885
4,Tombamento,6351
5,Colisão com objeto,5109
6,Colisão frontal,4739
7,Queda de ocupante de veículo,3450
8,Atropelamento de Pedestre,3057
9,Colisão lateral sentido oposto,2152
